# EDA 04 — Revenus des Ménages (INSEE Filosofi IRIS 2021)
**Source** : INSEE — Base IRIS Filosofi 2021  
**Bronze path** : `data/bronze/revenus/date=YYYY-MM-DD/part-0.parquet`  
**Scope** : Zones IRIS parisiennes (~300 zones)

## Schéma Bronze
| Colonne | Type | Description |
|---|---|---|
| `iris_code` | str | Code IRIS (9 caractères) |
| `commune_code` | str | Code INSEE commune |
| `arrondissement` | int | 1-20 |
| `median_income` | float | Revenu médian annuel (€) |
| `gini_coefficient` | float | Coefficient de Gini (inégalités) |
| `poverty_rate` | float | Taux de pauvreté (%) |
| `latitude` / `longitude` | float | Centroïde IRIS |


In [ ]:
import os, glob
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

BRONZE_ROOT = os.path.abspath("../../data/bronze")
rev_files = sorted(glob.glob(f"{BRONZE_ROOT}/revenus/**/*.parquet", recursive=True))
print(f"Fichiers revenus trouvés : {len(rev_files)}")


In [ ]:
if rev_files:
    df = pd.concat([pd.read_parquet(f) for f in rev_files], ignore_index=True)
else:
    rng = np.random.default_rng(42)
    n_iris = 300
    arrondissements = np.repeat(np.arange(1, 21), 15)[:n_iris]
    base_income = {arr: 30000 + (21-arr)*1500 + rng.normal(0, 3000) for arr in range(1, 21)}
    df = pd.DataFrame({
        "iris_code":        [f"751{arr:02d}{i:04d}" for arr, i in zip(arrondissements, range(n_iris))],
        "commune_code":     [f"751{arr:02d}" for arr in arrondissements],
        "arrondissement":   arrondissements,
        "median_income":    [max(15000, base_income[arr] + rng.normal(0, 4000)) for arr in arrondissements],
        "gini_coefficient": rng.uniform(0.25, 0.50, n_iris),
        "poverty_rate":     rng.uniform(5, 35, n_iris),
        "year":             2021,
        "latitude":         48.8566 + rng.uniform(-0.07, 0.07, n_iris),
        "longitude":        2.3522  + rng.uniform(-0.08, 0.08, n_iris),
        "ingested_at":      pd.Timestamp("now", tz="UTC"),
    })
    print("⚠️  Fichiers Bronze absents — démonstration sur données synthétiques")

print(f"Shape : {df.shape}")
df.describe().round(2)


In [ ]:
# ── Distribution des revenus médians ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].hist(df["median_income"].dropna() / 1000, bins=40, color="#4CAF50", edgecolor="white")
axes[0].set_title("Distribution des revenus médians (IRIS)")
axes[0].set_xlabel("Revenu médian annuel (k€)")
axes[0].axvline(df["median_income"].median()/1000, color="red", linestyle="--",
                label=f"Médiane : {df['median_income'].median()/1000:.1f} k€")
axes[0].legend()

axes[1].scatter(df["gini_coefficient"], df["poverty_rate"],
                c=df["median_income"], cmap="RdYlGn", alpha=0.6, s=30)
axes[1].set_xlabel("Coefficient de Gini")
axes[1].set_ylabel("Taux de pauvreté (%)")
axes[1].set_title("Gini vs Taux de pauvreté (couleur = revenu médian)")
sm = plt.cm.ScalarMappable(cmap="RdYlGn", norm=plt.Normalize(df["median_income"].min(), df["median_income"].max()))
plt.colorbar(sm, ax=axes[1], label="Revenu médian (€)")
plt.tight_layout()
plt.show()


In [ ]:
# ── Revenu médian par arrondissement ─────────────────────────────────────────
arr_income = (
    df.groupby("arrondissement")
    .agg(revenu_median=("median_income","median"),
         gini_moyen=("gini_coefficient","mean"),
         pauvrete_moy=("poverty_rate","mean"),
         nb_iris=("iris_code","count"))
    .sort_values("revenu_median", ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = plt.cm.RdYlGn(np.linspace(0.1, 0.9, len(arr_income)))
axes[0].bar(arr_income.index.astype(str), arr_income["revenu_median"]/1000, color=colors)
axes[0].set_xlabel("Arrondissement")
axes[0].set_ylabel("Revenu médian (k€/an)")
axes[0].set_title("Revenu médian par arrondissement")
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45)

axes[1].bar(arr_income.index.astype(str), arr_income["pauvrete_moy"],
            color=plt.cm.Reds(np.linspace(0.3, 0.9, len(arr_income))))
axes[1].set_xlabel("Arrondissement")
axes[1].set_ylabel("Taux de pauvreté moyen (%)")
axes[1].set_title("Taux de pauvreté moyen par arrondissement")
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.show()
display(arr_income.round(2))


In [ ]:
# ── Corrélations revenus / inégalités ────────────────────────────────────────
corr_data = df[["median_income","gini_coefficient","poverty_rate"]].dropna()
corr_matrix = corr_data.corr()

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr_matrix, cmap="RdBu", vmin=-1, vmax=1)
ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(corr_matrix.columns, rotation=45)
ax.set_yticklabels(corr_matrix.index)
plt.colorbar(im, ax=ax)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{corr_matrix.iloc[i,j]:.2f}", ha="center", va="center", fontsize=12)
ax.set_title("Matrice de corrélation — Indicateurs revenus")
plt.tight_layout()
plt.show()


## Conclusions EDA
- Fort gradient Est-Ouest des revenus : les arrondissements Ouest (16e, 8e) ont des revenus 2-3x supérieurs aux arrondissements populaires de l'Est (19e, 20e).
- Corrélation négative forte entre revenu médian et taux de pauvreté.
- Le coefficient de Gini varie peu entre IRIS d'un même arrondissement : les inégalités sont surtout inter-arrondissements.
- Données millésime 2021 — intégration en Silver comme indicateur de contexte socio-économique.
